# Day 5 — PySpark Practice Questions: All Joins

| # | Difficulty | Join Type | Topic |
|---|-----------|-----------|-------|
| 1 | 🟢 Easy   | INNER     | Orders with customer names |
| 2 | 🟢 Easy   | LEFT + isNull | Customers with no orders |
| 3 | 🟡 Medium | LEFT + groupBy | Total orders + revenue per customer |
| 4 | 🟢 Easy   | SELF      | Employee → manager hierarchy |
| 5 | 🟡 Medium | FULL OUTER | Unmatched rows on both sides |
| 6 | 🟢 Easy   | CROSS     | All customer–product combinations |

In [ ]:
import os
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day5_PySpark_PQ') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

# ── customers ────────────────────────────────────────────────────────────────
customers_data = [
    (1, 'Alice', 'alice@example.com', 'New York'),
    (2, 'Bob',   'bob@example.com',   'Chicago'),
    (3, 'Carol', 'carol@example.com', 'Houston'),
    (4, 'Dave',  'dave@example.com',  'Phoenix'),   # no orders
    (5, 'Eve',   'eve@example.com',   'Seattle'),
]
customers_schema = StructType([
    StructField('customer_id', IntegerType(), False),
    StructField('name',        StringType(),  True),
    StructField('email',       StringType(),  True),
    StructField('city',        StringType(),  True),
])
df_customers = spark.createDataFrame(customers_data, schema=customers_schema)

# ── orders (cust 99 = orphan, Dave = no orders) ───────────────────────────────
orders_data = [
    (1,  1,  250.00, 'completed', '2024-01-05'),
    (2,  1,  125.50, 'pending',   '2024-01-12'),
    (3,  2,   89.99, 'completed', '2024-01-07'),
    (4,  3,  450.00, 'completed', '2024-01-10'),
    (5,  3,  210.00, 'cancelled', '2024-01-14'),
    (6,  5,  320.00, 'completed', '2024-01-08'),
    (7, 99,   75.00, 'pending',   '2024-01-15'),
]
orders_schema = StructType([
    StructField('order_id',    IntegerType(), False),
    StructField('customer_id', IntegerType(), True),
    StructField('amount',      DoubleType(),  True),
    StructField('status',      StringType(),  True),
    StructField('order_date',  StringType(),  True),
])
df_orders = spark.createDataFrame(orders_data, schema=orders_schema)

# ── employees ─────────────────────────────────────────────────────────────────
employees_data = [
    (1, 'Sarah', 'CEO',            None),
    (2, 'Tom',   'VP Engineering',    1),
    (3, 'Priya', 'VP Marketing',      1),
    (4, 'Alice', 'Engineer',          2),
    (5, 'Bob',   'Engineer',          2),
    (6, 'Carol', 'Designer',          3),
    (7, 'Dave',  'Analyst',           3),
]
employees_schema = StructType([
    StructField('emp_id',     IntegerType(), False),
    StructField('name',       StringType(),  True),
    StructField('role',       StringType(),  True),
    StructField('manager_id', IntegerType(), True),
])
df_employees = spark.createDataFrame(employees_data, schema=employees_schema)

# ── products ──────────────────────────────────────────────────────────────────
products_data = [
    (1, 'Widget A',  'Hardware',  29.99),
    (2, 'Widget B',  'Hardware',  49.99),
    (3, 'Service X', 'Software',  99.00),
    (4, 'Service Y', 'Software', 149.00),
]
products_schema = StructType([
    StructField('product_id',   IntegerType(), False),
    StructField('product_name', StringType(),  True),
    StructField('category',     StringType(),  True),
    StructField('price',        DoubleType(),  True),
])
df_products = spark.createDataFrame(products_data, schema=products_schema)

print('DataFrames ready — customers:', df_customers.count(),
      '| orders:', df_orders.count(),
      '| employees:', df_employees.count(),
      '| products:', df_products.count())

---
# Question 1 🟢 — INNER JOIN: Orders with Customer Names

## Concept Warm-up

In [ ]:
# Warm-up 1: basic join syntax — string 'on' deduplicates the key column
df_orders.join(df_customers, on='customer_id', how='inner').printSchema()

In [ ]:
# Warm-up 2: count before and after to see what got dropped
print('orders rows:', df_orders.count())        # 7 (including orphan)
print('inner join rows:', df_orders.join(df_customers, on='customer_id', how='inner').count())

In [ ]:
# Warm-up 3: .select() after join to pick columns
(
    df_orders
    .join(df_customers, on='customer_id', how='inner')
    .select('order_id', 'name', 'amount')
    .show(3)
)

## Problem

Return all orders that have a matching customer.  
Columns: `order_id`, `name` (customer), `city`, `amount`, `status`.  
Sort by `order_id` ascending.

**Expected:** 6 rows — orphan order (customer_id=99) must be excluded.

In [ ]:
# YOUR CODE HERE
df_q1 = None

# df_q1.show()

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────
df_q1 = (
    df_orders
    .join(df_customers, on='customer_id', how='inner')
    .select('order_id', 'name', 'city', 'amount', 'status')
    .orderBy('order_id')
)
df_q1.show()
assert df_q1.count() == 6, f'Expected 6 rows, got {df_q1.count()}'
# Orphan order (order_id=7) must be absent
assert df_q1.filter(F.col('order_id') == 7).count() == 0, 'Orphan order should be excluded'
print('Q1 PASSED')

---
# Question 2 🟢 — LEFT JOIN + isNull: Customers with No Orders

## Concept Warm-up

In [ ]:
# Warm-up 1: LEFT JOIN — all customers, including Dave who has no orders
(
    df_customers
    .join(df_orders, on='customer_id', how='left')
    .select('customer_id', 'name', 'order_id', 'amount')
    .orderBy('customer_id', 'order_id')
    .show()
)

In [ ]:
# Warm-up 2: isNull() is correct — == None does NOT work in PySpark
df_left = df_customers.join(df_orders, on='customer_id', how='left')

# Wrong way — always returns 0
print('Wrong (== None):', df_left.filter(df_left.order_id == None).count())

# Correct way
print('Correct (.isNull()):', df_left.filter(F.col('order_id').isNull()).count())

In [ ]:
# Warm-up 3: filter after join
(
    df_customers
    .join(df_orders, on='customer_id', how='left')
    .filter(F.col('order_id').isNull())
    .select('name')
    .show()
)

## Problem

Find all customers who have placed **zero orders**.  
Columns: `customer_id`, `name`, `email`.  

**Expected:** 1 row — Dave (customer_id=4)

In [ ]:
# YOUR CODE HERE
df_q2 = None

# df_q2.show()

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────
df_q2 = (
    df_customers
    .join(df_orders, on='customer_id', how='left')
    .filter(F.col('order_id').isNull())
    .select('customer_id', 'name', 'email')
)
df_q2.show()
assert df_q2.count() == 1, f'Expected 1 row, got {df_q2.count()}'
assert df_q2.filter(F.col('name') == 'Dave').count() == 1, 'Dave should be in result'
print('Q2 PASSED')

---
# Question 3 🟡 — LEFT JOIN + GROUP BY: Revenue per Customer

## Concept Warm-up

In [ ]:
# Warm-up 1: F.count('order_id') counts non-NULL only → 0 for Dave
(
    df_customers
    .join(df_orders, on='customer_id', how='left')
    .groupBy('customer_id', 'name')
    .agg(F.count('order_id').alias('total_orders'))
    .orderBy('customer_id')
    .show()
)

In [ ]:
# Warm-up 2: F.sum returns NULL for empty groups — F.coalesce replaces with 0
(
    df_customers
    .join(df_orders, on='customer_id', how='left')
    .groupBy('customer_id', 'name')
    .agg(
        F.sum('amount').alias('raw_sum'),
        F.coalesce(F.sum('amount'), F.lit(0)).alias('safe_sum'),
    )
    .orderBy('customer_id')
    .show()
)

In [ ]:
# Warm-up 3: orderBy descending
(
    df_customers
    .join(df_orders, on='customer_id', how='left')
    .groupBy('customer_id', 'name')
    .agg(F.coalesce(F.sum('amount'), F.lit(0)).alias('total'))
    .orderBy('total', ascending=False)
    .show()
)

## Problem

For **every customer** (including those with no orders), compute:
- `total_orders` — number of orders (0 for Dave)
- `total_revenue` — sum of amounts (0.0 for Dave, not NULL)

Columns: `customer_id`, `name`, `total_orders`, `total_revenue`  
Sort by `total_revenue` descending.

**Expected:** 5 rows — Dave must show `total_orders=0, total_revenue=0`

In [ ]:
# YOUR CODE HERE
df_q3 = None

# df_q3.show()

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────
df_q3 = (
    df_customers
    .join(df_orders, on='customer_id', how='left')
    .groupBy('customer_id', 'name')
    .agg(
        F.count('order_id').alias('total_orders'),
        F.coalesce(F.sum('amount'), F.lit(0)).alias('total_revenue'),
    )
    .orderBy('total_revenue', ascending=False)
)
df_q3.show()
assert df_q3.count() == 5, f'Expected 5 rows, got {df_q3.count()}'
dave = df_q3.filter(F.col('name') == 'Dave').first()
assert dave['total_orders'] == 0,    f"Dave total_orders should be 0, got {dave['total_orders']}"
assert dave['total_revenue'] == 0.0, f"Dave total_revenue should be 0, got {dave['total_revenue']}"
print('Q3 PASSED')

---
# Question 4 🟢 — SELF JOIN: Employee → Manager

## Concept Warm-up

In [ ]:
# Warm-up 1: inspect employees
df_employees.show()

In [ ]:
# Warm-up 2: alias the same DataFrame — both 'e' and 'm' point to df_employees
emp = df_employees.alias('e')
mgr = df_employees.alias('m')
print('emp type:', type(emp))

# Must use F.col('alias.column') to reference specific side
# F.col('e.name') vs F.col('m.name')

In [ ]:
# Warm-up 3: INNER self join drops CEO (no manager_id) — see the difference
emp = df_employees.alias('e')
mgr = df_employees.alias('m')

# INNER — CEO (Sarah) dropped
inner_result = emp.join(mgr, F.col('e.manager_id') == F.col('m.emp_id'), how='inner')
print('INNER rows (no CEO):', inner_result.count())

# LEFT — CEO kept with null manager
left_result = emp.join(mgr, F.col('e.manager_id') == F.col('m.emp_id'), how='left')
print('LEFT rows (CEO included):', left_result.count())

## Problem

List **all employees** with their manager's name.  
The CEO (no manager) must still appear — show `null` for manager.  
Columns: `emp_id`, `employee`, `role`, `manager`.  
Sort by `emp_id`.

In [ ]:
# YOUR CODE HERE
df_q4 = None

# df_q4.show()

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────
emp = df_employees.alias('e')
mgr = df_employees.alias('m')

df_q4 = (
    emp.join(
        mgr,
        F.col('e.manager_id') == F.col('m.emp_id'),
        how='left'
    )
    .select(
        F.col('e.emp_id'),
        F.col('e.name').alias('employee'),
        F.col('e.role'),
        F.col('m.name').alias('manager'),
    )
    .orderBy('e.emp_id')
)
df_q4.show()
assert df_q4.count() == 7, f'Expected 7 rows, got {df_q4.count()}'
sarah = df_q4.filter(F.col('employee') == 'Sarah').first()
assert sarah['manager'] is None, 'Sarah (CEO) should have null manager'
alice = df_q4.filter(F.col('employee') == 'Alice').first()
assert alice['manager'] == 'Tom', f"Alice's manager should be Tom, got {alice['manager']}"
print('Q4 PASSED')

---
# Question 5 🟡 — FULL OUTER JOIN: Unmatched Rows on Both Sides

## Concept Warm-up

In [ ]:
# Warm-up 1: outer join returns everything from both sides
(
    df_customers
    .join(df_orders, on='customer_id', how='outer')
    .select('customer_id', 'name', 'order_id', 'amount')
    .orderBy('customer_id', 'order_id')
    .show()
)

In [ ]:
# Warm-up 2: filter for rows where EITHER side is null
(
    df_customers
    .join(df_orders, on='customer_id', how='outer')
    .filter(F.col('order_id').isNull() | F.col('name').isNull())
    .select('customer_id', 'name', 'order_id', 'amount')
    .show()
)

In [ ]:
# Warm-up 3: F.when().otherwise() for labelling
from pyspark.sql.functions import when
(
    df_customers
    .join(df_orders, on='customer_id', how='outer')
    .filter(F.col('order_id').isNull() | F.col('name').isNull())
    .select(
        'name', 'order_id',
        when(F.col('name').isNull(),     'Orphan order')
        .when(F.col('order_id').isNull(),'No orders')
        .alias('reason')
    )
    .show()
)

## Problem

Find all **unmatched rows** — rows without a match on either side.  
Include unmatched customers (no orders) AND orphan orders (no customer).  
Add a `reason` column: `'Customer with no orders'` or `'Orphan order'`.  
Columns: `customer_name`, `order_id`, `amount`, `reason`.

**Expected:** 2 rows — Dave (no orders) + order_id=7 (customer_id=99)

In [ ]:
# YOUR CODE HERE
df_q5 = None

# df_q5.show()

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────
df_q5 = (
    df_customers
    .join(df_orders, on='customer_id', how='outer')
    .filter(
        F.col('order_id').isNull() | F.col('name').isNull()
    )
    .select(
        F.col('name').alias('customer_name'),
        'order_id',
        'amount',
        F.when(F.col('name').isNull(),     'Orphan order')
         .when(F.col('order_id').isNull(), 'Customer with no orders')
         .alias('reason')
    )
)
df_q5.show()
assert df_q5.count() == 2, f'Expected 2 rows, got {df_q5.count()}'
reasons = {r['reason'] for r in df_q5.collect()}
assert 'Orphan order' in reasons, 'Missing Orphan order row'
assert 'Customer with no orders' in reasons, 'Missing Customer with no orders row'
print('Q5 PASSED')

---
# Question 6 🟢 — CROSS JOIN: All Customer–Product Combinations

## Concept Warm-up

In [ ]:
# Warm-up 1: crossJoin syntax — no 'on' or 'how'
df_customers.crossJoin(df_products).count()   # 5 × 4 = 20

In [ ]:
# Warm-up 2: schema after cross join — all columns from both DataFrames
df_customers.crossJoin(df_products).printSchema()

In [ ]:
# Warm-up 3: filter after cross join to get only 'Hardware' products
(
    df_customers
    .crossJoin(df_products)
    .filter(F.col('category') == 'Hardware')
    .select(F.col('name').alias('customer'), 'product_name', 'price')
    .orderBy('name', 'product_name')
    .show()
)

## Problem

Generate a **catalog** of all possible customer–product pairings.  
Columns: `customer` (customer name), `product_name`, `category`, `price`.  
Sort by `customer` ASC, then `product_name` ASC.

**Expected:** 20 rows (5 customers × 4 products)

In [ ]:
# YOUR CODE HERE
df_q6 = None

# df_q6.show()

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────────
df_q6 = (
    df_customers
    .crossJoin(df_products)
    .select(
        F.col('name').alias('customer'),
        'product_name',
        'category',
        'price',
    )
    .orderBy('customer', 'product_name')
)
df_q6.show()
assert df_q6.count() == 20, f'Expected 20 rows, got {df_q6.count()}'
customers_in_result = {r['customer'] for r in df_q6.collect()}
assert customers_in_result == {'Alice','Bob','Carol','Dave','Eve'}, 'All 5 customers must appear'
products_in_result = {r['product_name'] for r in df_q6.collect()}
assert products_in_result == {'Widget A','Widget B','Service X','Service Y'}, 'All 4 products must appear'
print('Q6 PASSED')

In [ ]:
spark.stop()
print('Spark stopped.')